In [1]:
import aiohttp
import asyncio
import nest_asyncio
import os
from dotenv import load_dotenv
from typing import List, Dict, Any, Tuple, Union
import json
from github import Github, Auth
from gidgethub.aiohttp import GitHubAPI
from gidgethub import GitHubException
import sys
import base64
from stamina import retry, retry_context
import re
from functools import partial
from textacy import preprocessing
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from qdrant_client.http.models import Distance, VectorParams
import uuid
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langgraph.graph import START, StateGraph
from langchain_core.documents import Document
from typing_extensions import List, TypedDict

import numpy as np
import matplotlib.pyplot as plt

# allow for multiple event loops within Jupyter
nest_asyncio.apply()

# load environment variables
load_dotenv()

import pickle

In [2]:
_URL_RE = re.compile(r'https?://[^\s\)\]"<>]+')


def _is_retriable(exc: Exception) -> bool:
    """Retry on transient network/server errors; skip deterministic 404/403."""
    if isinstance(exc, GitHubException):
        return exc.status_code not in (404, 403)
    return isinstance(exc, aiohttp.ClientError)


@retry(on=_is_retriable, attempts=3, wait_initial=0.5, wait_max=10.0)
async def get_root_markdown_files(gh: GitHubAPI, owner: str, repo: str) -> List[Dict[Any, Any]]:
    """
    Get all markdown files from the root directory of a repository.
    This includes README.md, CONTRIBUTING.md, CHANGELOG.md, etc.
    """
    try:
        # Get contents of root directory
        contents = await gh.getitem(f"/repos/{owner}/{repo}/contents/")

        # Filter for markdown files (case-insensitive)
        markdown_files = [
            file for file in contents
            if file['type'] == 'file' and file['name'].lower().endswith('.md')
        ]

        return markdown_files
    except GitHubException as e:
        print(f"Error fetching contents for {owner}/{repo}: {e}")
        return []


@retry(on=_is_retriable, attempts=3, wait_initial=0.5, wait_max=10.0)
async def fetch_markdown_content(gh: GitHubAPI, owner: str, repo: str, file_path: str):
    """Fetch content of a specific markdown file"""
    try:
        file_data = await gh.getitem(f"/repos/{owner}/{repo}/contents/{file_path}")
        content = base64.b64decode(file_data['content']).decode('utf-8')

        return {
            'name': file_data['name'],
            'path': file_data['path'],
            'size': file_data['size'],
            'content': content,
            'success': True
        }
    except GitHubException as e:
        return {
            'path': file_path,
            'success': False,
            'error': str(e)
        }


def count_urls(text: str) -> int:
    """Return the number of HTTP/HTTPS URLs found in a text string."""
    return len(_URL_RE.findall(text))


async def fetch_starred_repos_with_docs(max_repos: int = None, concurrent_tasks: int = 20) -> List[Dict[Any, Any]]:
    """
    Fetch all starred repositories for the authenticated GitHub user, then
    concurrently retrieve documentation for each repo using the following strategy:

      1. Try the /readme endpoint first (covers README.md, README.rst, etc.)
      2. If no README exists (404), fall back to fetching ALL markdown files
         found in the root directory of the repository.
      3. If neither exists, the repo is recorded with an empty docs list.

    All per-repo fetches run concurrently via asyncio.gather, so even a large
    starred list is handled as fast as the GitHub rate limit allows.

    Args:
        max_repos: Optional cap on the number of starred repos to process.
                   Defaults to None (fetch all starred repos).

    Returns:
        A list of dicts, one per repo, with keys:
            repo         - "owner/name"
            description  - repo description string
            stars        - stargazer count
            language     - primary language
            url          - HTML URL
            doc_source   - "readme" | "root_markdown" | None
            docs         - list of {name, path, size, content, url_count} dicts
    """
    oauth_token = os.getenv("GITHUB_TOKEN")

    semaphore = asyncio.Semaphore(concurrent_tasks)

    async with aiohttp.ClientSession() as session:
        gh = GitHubAPI(session, "markdown-fetcher", oauth_token=oauth_token)

        print("Fetching starred repositories...")
        starred_repos: List[Dict[Any, Any]] = []
        async for repo in gh.getiter("/user/starred", accept="application/vnd.github.mercy-preview+json"):
            starred_repos.append(repo)
            if max_repos and len(starred_repos) >= max_repos:
                break

        print(f"Found {len(starred_repos)} starred repositories")
        print("Fetching documentation for all repos concurrently...\n")

        async def fetch_repo_docs(repo: Dict[Any, Any]) -> Dict[Any, Any]:
            owner = repo["owner"]["login"]
            name = repo["name"]
            full_name = repo["full_name"]
            topics: List[str] = repo["topics"]

            base = {
                "repo": full_name,
                "description": repo.get("description"),
                "topics": topics,
                "stars": repo.get("stargazers_count"),
                "language": repo.get("language"),
                "url": repo.get("html_url"),
            }

            try:
                async for attempt in retry_context(
                    on=_is_retriable, attempts=3, wait_initial=0.5, wait_max=10.0
                ):
                    with attempt:
                        readme_data = await gh.getitem(f"/repos/{owner}/{name}/readme")
                content = base64.b64decode(readme_data["content"]).decode("utf-8")
                return {
                    **base,
                    "doc_source": "readme",
                    "docs": [
                        {
                            "name": readme_data["name"],
                            "path": readme_data["path"],
                            "size": readme_data["size"],
                            "content": content,
                            "url_count": count_urls(content),  # ← NEW
                        }
                    ],
                }
            except GitHubException as e:
                if e.status_code != 404:
                    print(f"  Warning: unexpected error fetching README for {full_name}: {e}")

            markdown_files = await get_root_markdown_files(gh, owner, name)
            if markdown_files:
                tasks = [
                    fetch_markdown_content(gh, owner, name, f["path"])
                    for f in markdown_files
                ]
                file_results = await asyncio.gather(*tasks)
                return {
                    **base,
                    "doc_source": "root_markdown",
                    "docs": [
                        {**r, "url_count": count_urls(r["content"])}  # ← NEW
                        for r in file_results if r.get("success")
                    ],
                }

            return {**base, "doc_source": None, "docs": []}

        async def fetch_repo_docs_throttled(repo: Dict[Any, Any]) -> Dict[Any, Any]:
            async with semaphore:
                return await fetch_repo_docs(repo)

        results: List[Dict[Any, Any]] = await asyncio.gather(
            *[fetch_repo_docs_throttled(repo) for repo in starred_repos]
        )

        with_readme = [r for r in results if r["doc_source"] == "readme"]
        with_md = [r for r in results if r["doc_source"] == "root_markdown"]
        no_docs = [r for r in results if r["doc_source"] is None]

        print(f"\n{'='*70}")
        print("DOCUMENTATION FETCH SUMMARY")
        print(f"{'='*70}")
        print(f"Total repos processed : {len(results)}")
        print(f"  README found        : {len(with_readme)}")
        print(f"  Root markdown files : {len(with_md)}")
        print(f"  No docs found       : {len(no_docs)}")
        if gh.rate_limit:
            print(f"Rate limit remaining  : {gh.rate_limit.remaining}")

        return results

In [3]:
MAX_REPOS = 3000

starred_repo_data = asyncio.run(fetch_starred_repos_with_docs(MAX_REPOS))

Fetching starred repositories...
Found 2085 starred repositories
Fetching documentation for all repos concurrently...


DOCUMENTATION FETCH SUMMARY
Total repos processed : 2085
  README found        : 2072
  Root markdown files : 1
  No docs found       : 12
Rate limit remaining  : 2789


In [6]:
starred_repo_data[10]

{'repo': 'facebook/pyrefly',
 'description': 'A fast type checker and language server for Python',
 'topics': ['code-quality',
  'contributions-welcome',
  'good-first-issue',
  'hacktoberfest',
  'ide',
  'language-server',
  'lsp',
  'python',
  'rust',
  'type-check',
  'type-checker',
  'typecheck',
  'typechecker',
  'types',
  'typing'],
 'stars': 5460,
 'language': 'Rust',
 'url': 'https://github.com/facebook/pyrefly',
 'doc_source': 'readme',
 'docs': [{'name': 'README.md',
   'path': 'README.md',
   'size': 7804,
   'content': '# Pyrefly: A fast type checker and language server for Python with powerful IDE features\n\n[![pyrefly](https://img.shields.io/endpoint?url=https://pyrefly.org/badge.json)](https://github.com/facebook/pyrefly)\n[![PyPI](https://img.shields.io/pypi/dm/pyrefly?color=blue&label=pypi)](https://pypi.python.org/pypi/pyrefly)\n[![Visual Studio Marketplace](https://img.shields.io/visual-studio-marketplace/d/meta.pyrefly?color=blue&label=VS%20Code)](https://mark

In [7]:
with open("../data/cached/github_data2.pkl", "wb") as f:
    pickle.dump(starred_repo_data, f)